# DSAR × Lakeflow Declarative Pipelines · 00b · Incremental landing + 2nd DSAR wave

**Run AFTER the initial cycle** (`00` → pipeline → `02`) to demonstrate the pipeline
running **incrementally** on newly-arriving files, followed by a **second erasure
wave** that mixes brand-new and older subjects.

It does two things:

1. **Lands a NEW batch of JSON files** into `…/raw_user/landing/incremental/` — 8
   brand-new customers `U900001`–`U900008` (ids far above the initial range, so "new"
   is obvious). When you next run the pipeline **incrementally** (normal Start, no full
   refresh), Auto Loader ingests only these new files.
   *(These are the same records committed to the repo under
   `sample_data/incremental_batch_1/` — you can instead upload those files to the
   volume by hand; see that folder's README.)*
2. **Enqueues a 2nd DSAR wave** — a deliberate mix:
   - some requests target the **newly-arrived** subjects, and
   - some target **older, pre-existing** subjects not in wave 1.

> Batch notebook — **Run all**. Point `catalog`/`schema`/`volume` at the **same**
> objects you ran the initial cycle against (for the CDC variant, its `..._cdc`
> schema). Re-runnable: appends another file + another wave, continuing the numbering.


## 0. Configuration


In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "dkushari_uc", "1 Catalog")
dbutils.widgets.text("schema", "allegiant_air_sdp_dsar", "2 Schema (match the cycle you ran)")
dbutils.widgets.text("volume", "raw_user", "3 Landing volume")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA  = dbutils.widgets.get("schema").strip()
VOLUME  = dbutils.widgets.get("volume").strip()
FQ      = f"{CATALOG}.{SCHEMA}"
INCREMENTAL = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/landing/incremental"
dbutils.fs.mkdirs(INCREMENTAL)
print("Incremental landing:", INCREMENTAL)


## 1. Land a NEW batch of files (8 new customers, U900001–U900008)

These match the committed `sample_data/incremental_batch_1/` files exactly. Written
as a new JSON file so Auto Loader treats it as a fresh arrival.


In [ ]:
import json as _json, datetime

FIRST = ["Robin","Skyler","Dakota","Reese","Phoenix","Sage","Rowan","Emerson"]
LAST  = ["Marsh","Blaine","Fischer","Okafor","Rivas","Larsen","Ibarra","Novak"]
now = datetime.datetime.utcnow().replace(microsecond=0)

records, new_emails = [], []
for i in range(8):
    uid   = f"U{900001+i:06d}"                     # clearly NEW ids
    first, last = FIRST[i], LAST[i]
    email = f"{first.lower()}.{last.lower()}{900001+i}@example.com"
    full  = f"{first} {last}"
    new_emails.append(email)
    for evt in range(3):
        rev = round(100 + i*13.5 + evt*7.25, 2)
        ets = (now - datetime.timedelta(hours=evt)).strftime("%Y-%m-%dT%H:%M:%S")
        its = (now + datetime.timedelta(minutes=evt)).strftime("%Y-%m-%dT%H:%M:%S")
        pj  = _json.dumps({"contact":{"email":email,"name":full},
                           "loyalty":{"tier":"silver","ltv":rev}}, separators=(",",":"))
        records.append({"event_id":f"{uid}-{evt}","user_id":uid,"email":email,
                        "full_name":full,"profile_json":pj,"revenue":rev,
                        "event_ts":ets,"_ingest_ts":its})

# unique filename per run so re-runs land distinct files
stamp = now.strftime("%Y%m%d%H%M%S")
target = f"{INCREMENTAL}/incremental_{stamp}.json"
with open("/" + target.split(":",1)[1] if ":" in target else target, "w") as fh:
    for r in records:
        fh.write(_json.dumps(r, separators=(",",":")) + "\n")

print(f"Landed {len(records)} new events ({len(new_emails)} new customers) -> {target}")
print("New subjects:", new_emails)


## 2. Enqueue the 2nd DSAR wave — mix of NEW and OLD subjects

4 new PENDING requests, continuing the `REQ-xxx` numbering:

| # | Subject | Type | Proves |
|---|---------|------|--------|
| a | a **NEW** customer | DELETE | erase brand-new incremental data (tables + files) |
| b | a **NEW** customer | OBFUSCATE | redact brand-new incremental data |
| c | an **OLD** customer (pre-existing, not in wave 1) | DELETE | erase data flowing since the initial load |
| d | an **OLD** customer | OBFUSCATE | redact long-standing data |

**Old** subjects are chosen from the pre-existing population, **excluding** anyone
already the target of a request **and** excluding already-redacted rows (whose email
is the token) — so `02` resolves a real, un-erased email.


In [ ]:
from pyspark.sql import functions as F

TOKEN = "***REDACTED***"

def _req_num(rid):
    try: return int(str(rid).split("-")[1])
    except Exception: return 0

existing_reqs     = spark.table(f"{FQ}.dsar_request")
already_requested = {r["subject_email"] for r in existing_reqs.select("subject_email").collect()}
next_n = max([_req_num(r["request_id"]) for r in existing_reqs.select("request_id").collect()] + [0]) + 1

# OLD candidates: pre-existing rows in raw_user, real (non-token) email, not the new
# batch, and not already requested. Read from the raw_user TABLE (populated by the
# pipeline), which is where 02 resolves email->user_id.
old_candidates = [r["email"] for r in (
    spark.table(f"{FQ}.raw_user")
         .where((F.col("user_id") < "U900000") & (F.col("email") != TOKEN))
         .select("email").distinct().orderBy("email").limit(2000).collect()
) if r["email"] not in already_requested][:2]

new_pick = [e for e in new_emails if e not in already_requested][:2]

assert len(new_pick) >= 2, "need >=2 new subjects not already requested"
assert len(old_candidates) >= 2, "no un-requested, un-erased OLD customers left; use a fresh schema"

wave = [
    (new_pick[0],       "DELETE",    "new"),
    (new_pick[1],       "OBFUSCATE", "new"),
    (old_candidates[0], "DELETE",    "old"),
    (old_candidates[1], "OBFUSCATE", "old"),
]
rows = [(f"REQ-{next_n+i:03d}", e, rt, "PENDING") for i, (e, rt, _) in enumerate(wave)]

df = (spark.createDataFrame(rows, "request_id string, subject_email string, request_type string, status string")
      .withColumn("request_date", F.current_date())
      .withColumn("deadline_date", F.date_add(F.current_date(), 45)))
df.write.mode("append").saveAsTable(f"{FQ}.dsar_request")

print("2nd DSAR wave enqueued:")
for (rid, e, rt, _st), (_e, _rt, cohort) in zip(rows, wave):
    print(f"  {rid}  {e:<40} {rt:<9} ({cohort})")
print("\nFull dsar_request queue now:")
display(spark.table(f"{FQ}.dsar_request").orderBy("request_id"))


## 3. Next — run the incremental cycle

1. **Run the pipeline incrementally** (same pipeline from `01`/`01b`): just **Start**
   (do NOT full-refresh). Auto Loader ingests only the new file; the new `U9000xx`
   customers appear in `silver_user` / `gold_user`.
2. **Run `02_erasure`** again (dry-run then live). It picks up the 2nd wave and erases
   both new and old subjects across every layer **and scrubs the volume files**, then
   refreshes gold. The pipeline never stops (`skipChangeCommits`).

See `usage.md` **Part B** for the full runbook.
